## Download Dataset

In [ ]:
from toolviper.utils.data import download

download(file="Antennae_North.cal.lsrk.ps.zarr")

## Setup Dask Cluster

In [ ]:
# import dask
# dask.config.set(scheduler="synchronous")

from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## Inspect Processing Set

In [ ]:
from xradio.measurement_set import open_processing_set
import pandas as pd

pd.options.display.max_colwidth = 100

intents = ["OBSERVE_TARGET#ON_SOURCE"]
ps_store = "Antennae_North.cal.lsrk.ps.zarr"
ps = open_processing_set(ps_store, scan_intents=intents)
ps.xr_ps.summary()

In [ ]:
ps["Antennae_North.cal.lsrk_34"]

## Run Cube Imaging Niter 0

In [ ]:
%load_ext autoreload
%autoreload 2

from xradio.measurement_set import open_processing_set
import pandas as pd

pd.options.display.max_colwidth = 100

scan_intents = ["OBSERVE_TARGET#ON_SOURCE"]
ps_store = "Antennae_North.cal.lsrk.ps.zarr"
ps = open_processing_set(ps_store, scan_intents=scan_intents)

ps_store = "Antennae_North.cal.lsrk.ps.zarr"
image_name = "Antennae_North_Cube.img.zarr"

import numpy as np
import os

grid_params = {}
grid_params["image_size"] = [500, 500]
grid_params["cell_size"] = np.array([-0.13, 0.13]) * np.pi / (180 * 3600)
grid_params["fft_padding"] = 1.0

combined_field_and_source_xds = ps.xr_ps.get_combined_field_and_source_xds()
center_field_name = combined_field_and_source_xds.attrs["center_field_name"]
grid_params["phase_direction"] = (
    combined_field_and_source_xds.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=center_field_name
    )
)
# ps['Antennae_North.cal.lsrk_34'].xr_ms.get_field_and_source_xds().FIELD_PHASE_CENTER_DIRECTION.isel(field_name=0)

spectral_params = {}
frequency_coord = ps["Antennae_North.cal.lsrk_34"].frequency
spectral_params["n_chunks"] = 60

polarization_params = {}

data_variables = [
    "sky",
    "point_spread_function",
    "primary_beam",
    "visibility",
]  # "visibility_normalization", "uv_sampling_normalization"

os.system("rm -rf " + image_name)
# n_chunks = 60
from astroviper.distributed.imaging.cube_imaging_niter0 import cube_imaging_niter0

polarization_coord = ps["Antennae_North.cal.lsrk_34"].polarization

cube_imaging_niter0(
    ps_store,
    image_name,
    grid_params,
    polarization_coord=polarization_coord,
    frequency_coord=frequency_coord,
    n_chunks=None,
    data_variables=data_variables,
)

## Inspect Image

In [ ]:
import xarray as xr

img_xds = xr.open_zarr("Antennae_North_Cube.img.zarr")
img_xds

In [ ]:
import matplotlib.pyplot as plt

# %matplotlib widget
plt.figure()
img_xds.POINT_SPREAD_FUNCTION.isel(polarization=1, frequency=82).plot(
    cmap="viridis", vmin=0.0
)
plt.figure()
img_xds.PRIMARY_BEAM.isel(polarization=0, frequency=82).plot()
plt.figure()
# img_xds.SKY.max(dim="frequency").isel(polarization=0).plot(cmap='viridis',vmin=0.0)
img_xds.SKY.isel(polarization=0, frequency=82).plot(cmap="viridis", vmin=0.0)

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact, fixed


def plot_astroviper_vs_casa_weights_interactive(ms_xdt, freq_idx):
    print()
    import matplotlib.pyplot as plt

    plt.figure()
    fig = plt.figure(figsize=(10, 8))
    img_xds.SKY.isel(polarization=0, frequency=freq_idx).plot(cmap="viridis", vmin=0.0)
    plt.show()
    plt.close(fig)

    plt.figure()
    fig = plt.figure(figsize=(10, 8))
    img_xds.POINT_SPREAD_FUNCTION.real.isel(polarization=0, frequency=freq_idx).plot(
        cmap="viridis", vmin=0.0
    )
    plt.show()
    plt.close(fig)


interact(
    plot_astroviper_vs_casa_weights_interactive,
    ms_xdt=fixed(img_xds),
    freq_idx=widgets.IntSlider(
        min=0, max=165, step=1, value=2, description="freq_chan"
    ),
)